Date: 16/10/2024 <br/>
Desc: To find minimum number of reliably rcvd packets per time window for thresholding method of detecting anomaly.

# Imports

In [1]:
import pandas as pd # for data manipulation 
import numpy as np
# import matplotlib.pyplot as plt # for drawing graphs
import os, glob
from multiprocessing.pool import Pool
from itertools import repeat

# Find Min From Processed Data Using Single Thread

In [3]:
import pandas as pd # for data manipulation 
import numpy as np
# import matplotlib.pyplot as plt # for drawing graphs
import os, glob
from multiprocessing.pool import Pool
from itertools import repeat

# Load the data
DATASET_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_processed"
RELIABILITY_TH = 0.999
NUM_PROCS = 32

usi_list = [10, 20, 66.7, 100]
usi_min_num_reliable = []

def load_num_reliable(scenario, reliability_th, run_num_filter=None):
    # print(scenario)
    dl_df_list = []
    ul_df_list = []
    vid_df_list = []
    # Downlink
    dl_csv_files = glob.glob(os.path.join(scenario, "*_Downlink_Throughput.csv"))
    if run_num_filter != None:
        dl_csv_files = [file for file in dl_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in dl_csv_files:
        dl_df_list.append(pd.read_csv(file))
    # Uplink
    ul_csv_files = glob.glob(os.path.join(scenario, "*_Uplink_Throughput.csv"))
    if run_num_filter != None:
        ul_csv_files = [file for file in ul_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in ul_csv_files:
        ul_df_list.append(pd.read_csv(file))
    # Video
    vid_csv_files = glob.glob(os.path.join(scenario, "*_Video_Throughput.csv"))
    if run_num_filter != None:
        vid_csv_files = [file for file in vid_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in vid_csv_files:
        vid_df_list.append(pd.read_csv(file))
    
    dl_df = pd.concat(dl_df_list)
    ul_df = pd.concat(ul_df_list)
    vid_df = pd.concat(vid_df_list)

    # Remove data where measured reliability < threshold.Take only data from t=1 onwards
    dl_df = dl_df.loc[(dl_df["Measured_Reliability"] >= reliability_th) & (dl_df["Time"] >= 1)]
    ul_df = ul_df.loc[(ul_df["Measured_Reliability"] >= reliability_th) & (ul_df["Time"] >= 1)]
    vid_df = vid_df.loc[(vid_df["Measured_Reliability"] >= reliability_th) & (vid_df["Time"] >= 1)]

    dl_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    ul_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    vid_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    
    return (dl_df["Num_Reliable"].min(), ul_df["Num_Reliable"].min(), vid_df["Num_Reliable"].min(), dl_df["Num_Reliable"].max(), ul_df["Num_Reliable"].max(), vid_df["Num_Reliable"].max(),
            scenario)


for usi in usi_list:
    # usi_scenarios = [scenario for scenario in scenario_files if "_UAVSendingInterval-{}".format(usi) in scenario]
    usi_scenarios = glob.glob(DATASET_PATH + "/*_UAVSendingInterval-{}".format(usi))
    dl_min_list = []
    ul_min_list = []
    vid_min_list = []
    dl_max_list = []
    ul_max_list = []
    vid_max_list = []
    with Pool(NUM_PROCS) as pool:
        for result in pool.starmap(load_num_reliable, zip(usi_scenarios, repeat(RELIABILITY_TH))):
            print(result)
            dl_min_list.append(result[0])
            ul_min_list.append(result[1])
            vid_min_list.append(result[2])
            dl_max_list.append(result[3])
            ul_max_list.append(result[4])
            vid_max_list.append(result[5])
  
    dl_min_num_reliable = np.nanmin(dl_min_list)
    ul_min_num_reliable = np.nanmin(ul_min_list)
    vid_min_num_reliable = np.nanmin(vid_min_list)

    dl_max_num_reliable = np.nanmax(dl_max_list)
    ul_max_num_reliable = np.nanmax(ul_max_list)
    vid_max_num_reliable = np.nanmax(vid_max_list)

    usi_min_num_reliable.append({"USI": usi, "DL_Min_Num_Reliable": dl_min_num_reliable, "UL_Min_Num_Reliable": ul_min_num_reliable, "VID_Min_Num_Reliable": vid_min_num_reliable, 
                               "DL_Max_Num_Reliable": dl_max_num_reliable, "UL_Max_Num_Reliable": ul_max_num_reliable, "VID_Max_Num_Reliable": vid_max_num_reliable})

usi_min_num_reliable_df = pd.DataFrame(usi_min_num_reliable)
usi_min_num_reliable_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/data_manual_num_reliable_min_max_train_17102024.csv")

(50, 772, 172, 50, 800, 173, '/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_processed/UAVSpeed-18_BitRate-39_Height-75_Distance-130_Modulation-QAM-16_UAVSendingInterval-10')
(50, 772, 172, 50, 800, 173, '/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_processed/UAVSpeed-18_BitRate-26_Height-105_Distance-150_Modulation-QAM-16_UAVSendingInterval-10')
(50, 772, 172, 50, 800, 173, '/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_processed/UAVSpeed-20_BitRate-39_Height-135_Distance-120_Modulation-QAM-16_UAVSendingInterval-10')
(50, 772, 172, 50, 800, 173, '/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_processed/UAVSpeed-20_BitRate-52_Height-75_Distance-80_Modulation-QA

# Using Multiprocessing

In [ ]:
# Load the data
DATASET_PATH = "/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_processed"
RELIABILITY_TH = 0.999
NUM_PROCS = 32
# scenario_files = glob.glob(DATASET_PATH + "/*")

usi_list = [10, 20, 66.7, 100]
usi_min_num_reliable = []

def load_num_reliable(scenario, reliability_th, run_num_filter=None):
    # print(scenario)
    dl_df_list = []
    ul_df_list = []
    vid_df_list = []
    # Downlink
    dl_csv_files = glob.glob(os.path.join(scenario, "*_Downlink_Throughput.csv"))
    if run_num_filter != None:
        dl_csv_files = [file for file in dl_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in dl_csv_files:
        dl_df_list.append(pd.read_csv(file))
    # Uplink
    ul_csv_files = glob.glob(os.path.join(scenario, "*_Uplink_Throughput.csv"))
    if run_num_filter != None:
        ul_csv_files = [file for file in ul_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in ul_csv_files:
        ul_df_list.append(pd.read_csv(file))
    # Video
    vid_csv_files = glob.glob(os.path.join(scenario, "*_Video_Throughput.csv"))
    if run_num_filter != None:
        vid_csv_files = [file for file in vid_csv_files if (file.split("/")[-1].split("_")[0] in run_num_filter)]
    for file in vid_csv_files:
        vid_df_list.append(pd.read_csv(file))
    
    dl_df = pd.concat(dl_df_list)
    ul_df = pd.concat(ul_df_list)
    vid_df = pd.concat(vid_df_list)

    # Remove data where measured reliability < threshold.Take only data from t=1 onwards
    dl_df = dl_df.loc[(dl_df["Measured_Reliability"] >= reliability_th) & (dl_df["Time"] >= 1)]
    ul_df = ul_df.loc[(ul_df["Measured_Reliability"] >= reliability_th) & (ul_df["Time"] >= 1)]
    vid_df = vid_df.loc[(vid_df["Measured_Reliability"] >= reliability_th) & (vid_df["Time"] >= 1)]

    dl_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    ul_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    vid_df.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
    
    return (dl_df["Num_Reliable"].min(), ul_df["Num_Reliable"].min(), vid_df["Num_Reliable"].min(), dl_df["Num_Reliable"].max(), ul_df["Num_Reliable"].max(), vid_df["Num_Reliable"].max())

for usi in usi_list:
    # usi_scenarios = [scenario for scenario in scenario_files if "_UAVSendingInterval-{}".format(usi) in scenario]
    usi_scenarios = glob.glob(DATASET_PATH + "/*_UAVSendingInterval-{}".format(usi))
    dl_min_list = []
    ul_min_list = []
    vid_min_list = []
    dl_max_list = []
    ul_max_list = []
    vid_max_list = []
    
    with Pool(NUM_PROCS) as pool:
        for result in pool.starmap(load_num_reliable, zip(usi_scenarios, repeat(RELIABILITY_TH))):
            print(result)
            dl_min_list.append(result[0])
            ul_min_list.append(result[1])
            vid_min_list.append(result[2])
            dl_max_list.append(result[3])
            ul_max_list.append(result[4])
            vid_max_list.append(result[5])
  
    dl_min_num_reliable = np.nanmin(dl_min_list)
    ul_min_num_reliable = np.nanmin(ul_min_list)
    vid_min_num_reliable = np.nanmin(vid_min_list)

    dl_max_num_reliable = np.nanmax(dl_max_list)
    ul_max_num_reliable = np.nanmax(ul_max_list)
    vid_max_num_reliable = np.nanmax(vid_max_list)

    usi_min_num_reliable.append({"USI": usi, "DL_Min_Num_Reliable": dl_min_num_reliable, "UL_Min_Num_Reliable": ul_min_num_reliable, "VID_Min_Num_Reliable": vid_min_num_reliable, 
                               "DL_Max_Num_Reliable": dl_max_num_reliable, "UL_Max_Num_Reliable": ul_max_num_reliable, "VID_Max_Num_Reliable": vid_max_num_reliable})

usi_min_num_reliable_df = pd.DataFrame(usi_min_num_reliable)
usi_min_num_reliable_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/data_manual_num_reliable_min_max_16102024.csv")

# Find Min From Compiled Processed Data

In [2]:
# Load compiled training dataset
uav_0_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_0_processed.csv")
uav_1_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_1_processed.csv")
uav_2_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_2_processed.csv")
uav_3_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_3_processed.csv")
uav_4_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_4_processed.csv")
uav_5_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_5_processed.csv")
uav_6_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_6_processed.csv")
uav_7_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_UAV_7_processed.csv")
ul_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_ul_processed.csv")
vid_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_new_vid_processed.csv")

In [2]:
# Load compiled testing dataset
uav_0_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_0_processed.csv")
uav_1_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_1_processed.csv")
uav_2_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_2_processed.csv")
uav_3_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_3_processed.csv")
uav_4_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_4_processed.csv")
uav_5_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_5_processed.csv")
uav_6_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_6_processed.csv")
uav_7_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_UAV_7_processed.csv")
ul_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_ul_processed.csv")
vid_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_no_int_test_99_vid_processed.csv")

In [3]:
RELIABILITY_TH = 0.999

# Remove data where measured reliability < threshold.Take only data from t=1 onwards
# uav_0_num_reliable_df_tmp = uav_0_num_reliable_df.loc[(uav_0_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_0_num_reliable_df["Time"] >= 1)]
# uav_1_num_reliable_df_tmp = uav_1_num_reliable_df.loc[(uav_1_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_1_num_reliable_df["Time"] >= 1)]
# uav_2_num_reliable_df_tmp = uav_2_num_reliable_df.loc[(uav_2_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_2_num_reliable_df["Time"] >= 1)]
# uav_3_num_reliable_df_tmp = uav_3_num_reliable_df.loc[(uav_3_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_3_num_reliable_df["Time"] >= 1)]
# uav_4_num_reliable_df_tmp = uav_4_num_reliable_df.loc[(uav_4_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_4_num_reliable_df["Time"] >= 1)]
# uav_5_num_reliable_df_tmp = uav_5_num_reliable_df.loc[(uav_5_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_5_num_reliable_df["Time"] >= 1)]
# uav_6_num_reliable_df_tmp = uav_6_num_reliable_df.loc[(uav_6_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_6_num_reliable_df["Time"] >= 1)]
# uav_7_num_reliable_df_tmp = uav_7_num_reliable_df.loc[(uav_7_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_7_num_reliable_df["Time"] >= 1)]
# ul_num_reliable_df_tmp = ul_num_reliable_df.loc[(ul_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (ul_num_reliable_df["Time"] >= 1)]
# vid_num_reliable_df_tmp = vid_num_reliable_df.loc[(vid_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (vid_num_reliable_df["Time"] >= 1)]

uav_0_num_reliable_df_tmp = uav_0_num_reliable_df.loc[(uav_0_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_0_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_0_num_reliable_df["Time"] >= 1)]
uav_1_num_reliable_df_tmp = uav_1_num_reliable_df.loc[(uav_1_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_1_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_1_num_reliable_df["Time"] >= 1)]
uav_2_num_reliable_df_tmp = uav_2_num_reliable_df.loc[(uav_2_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_2_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_2_num_reliable_df["Time"] >= 1)]
uav_3_num_reliable_df_tmp = uav_3_num_reliable_df.loc[(uav_3_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_3_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_3_num_reliable_df["Time"] >= 1)]
uav_4_num_reliable_df_tmp = uav_4_num_reliable_df.loc[(uav_4_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_4_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_4_num_reliable_df["Time"] >= 1)]
uav_5_num_reliable_df_tmp = uav_5_num_reliable_df.loc[(uav_5_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_5_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_5_num_reliable_df["Time"] >= 1)]
uav_6_num_reliable_df_tmp = uav_6_num_reliable_df.loc[(uav_6_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_6_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_6_num_reliable_df["Time"] >= 1)]
uav_7_num_reliable_df_tmp = uav_7_num_reliable_df.loc[(uav_7_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_7_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_7_num_reliable_df["Time"] >= 1)]
ul_num_reliable_df_tmp = ul_num_reliable_df.loc[(ul_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (ul_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (ul_num_reliable_df["Time"] >= 1)]
vid_num_reliable_df_tmp = vid_num_reliable_df.loc[(vid_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (vid_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (vid_num_reliable_df["Time"] >= 1)]

uav_0_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_1_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_2_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_3_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_4_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_5_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_6_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
uav_7_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
ul_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
vid_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)

/tmp/ipykernel_1508528/2454681008.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uav_0_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
/tmp/ipykernel_1508528/2454681008.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uav_1_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
/tmp/ipykernel_1508528/2454681008.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-ve

In [4]:
uav_0_num_reliable_df_tmp = uav_0_num_reliable_df_tmp.loc[(uav_0_num_reliable_df_tmp["Run_Num"] < 100)]
uav_1_num_reliable_df_tmp = uav_1_num_reliable_df_tmp.loc[(uav_1_num_reliable_df_tmp["Run_Num"] < 100)]
uav_2_num_reliable_df_tmp = uav_2_num_reliable_df_tmp.loc[(uav_2_num_reliable_df_tmp["Run_Num"] < 100)]
uav_3_num_reliable_df_tmp = uav_3_num_reliable_df_tmp.loc[(uav_3_num_reliable_df_tmp["Run_Num"] < 100)]
uav_4_num_reliable_df_tmp = uav_4_num_reliable_df_tmp.loc[(uav_4_num_reliable_df_tmp["Run_Num"] < 100)]
uav_5_num_reliable_df_tmp = uav_5_num_reliable_df_tmp.loc[(uav_5_num_reliable_df_tmp["Run_Num"] < 100)]
uav_6_num_reliable_df_tmp = uav_6_num_reliable_df_tmp.loc[(uav_6_num_reliable_df_tmp["Run_Num"] < 100)]
uav_7_num_reliable_df_tmp = uav_7_num_reliable_df_tmp.loc[(uav_7_num_reliable_df_tmp["Run_Num"] < 100)]
ul_num_reliable_df_tmp = ul_num_reliable_df_tmp.loc[(ul_num_reliable_df_tmp["Run_Num"] < 100)]
vid_num_reliable_df_tmp = vid_num_reliable_df_tmp.loc[(vid_num_reliable_df_tmp["Run_Num"] < 100)]

In [4]:
# Get min for gamma for each USI
usi_list = [10, 20, 66.7, 100]
ul_min_nr = []
uav_0_min_nr = []
uav_1_min_nr = []
uav_2_min_nr = []
uav_3_min_nr = []
uav_4_min_nr = []
uav_5_min_nr = []
uav_6_min_nr = []
uav_7_min_nr = []
vid_min_nr = []
for usi in usi_list:
    ul_usi_df = ul_num_reliable_df_tmp.loc[ul_num_reliable_df_tmp["USI"]==usi]
    ul_min_nr.append(ul_usi_df["Num_Reliable"].min())
    dl_df = uav_0_num_reliable_df_tmp.loc[uav_0_num_reliable_df_tmp["USI"]==usi]
    uav_0_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_1_num_reliable_df_tmp.loc[uav_1_num_reliable_df_tmp["USI"]==usi]
    uav_1_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_2_num_reliable_df_tmp.loc[uav_2_num_reliable_df_tmp["USI"]==usi]
    uav_2_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_3_num_reliable_df_tmp.loc[uav_3_num_reliable_df_tmp["USI"]==usi]
    uav_3_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_4_num_reliable_df_tmp.loc[uav_4_num_reliable_df_tmp["USI"]==usi]
    uav_4_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_5_num_reliable_df_tmp.loc[uav_5_num_reliable_df_tmp["USI"]==usi]
    uav_5_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_6_num_reliable_df_tmp.loc[uav_6_num_reliable_df_tmp["USI"]==usi]
    uav_6_min_nr.append(dl_df["Num_Reliable"].min())
    dl_df = uav_7_num_reliable_df_tmp.loc[uav_7_num_reliable_df_tmp["USI"]==usi]
    uav_7_min_nr.append(dl_df["Num_Reliable"].min())
    vid_df = vid_num_reliable_df_tmp.loc[vid_num_reliable_df_tmp["USI"]==usi]
    vid_min_nr.append(vid_df["Num_Reliable"].min())

num_reliable_df = pd.DataFrame([{"UAV_0_USI10_Min_Gamma": uav_0_min_nr[0], "UAV_1_USI10_Min_Gamma": uav_1_min_nr[0], "UAV_2_USI10_Min_Gamma": uav_2_min_nr[0], "UAV_3_USI10_Min_Gamma": uav_3_min_nr[0],
                                 "UAV_4_USI10_Min_Gamma": uav_4_min_nr[0], "UAV_5_USI10_Min_Gamma": uav_5_min_nr[0], "UAV_6_USI10_Min_Gamma": uav_6_min_nr[0], "UAV_7_USI10_Min_Gamma": uav_7_min_nr[0],
                                 "UL_USI10_Min_Gamma": ul_min_nr[0], "VID_USI10_Min_Gamma": vid_min_nr[0],
                                 "UAV_0_USI20_Min_Gamma": uav_0_min_nr[1], "UAV_1_USI20_Min_Gamma": uav_1_min_nr[1], "UAV_2_USI20_Min_Gamma": uav_2_min_nr[1], "UAV_3_USI20_Min_Gamma": uav_3_min_nr[1],
                                 "UAV_4_USI20_Min_Gamma": uav_4_min_nr[1], "UAV_5_USI20_Min_Gamma": uav_5_min_nr[1], "UAV_6_USI20_Min_Gamma": uav_6_min_nr[1], "UAV_7_USI20_Min_Gamma": uav_7_min_nr[1],
                                 "UL_USI20_Min_Gamma": ul_min_nr[1], "VID_USI20_Min_Gamma": vid_min_nr[1],
                                 "UAV_0_USI667_Min_Gamma": uav_0_min_nr[2], "UAV_1_USI667_Min_Gamma": uav_1_min_nr[2], "UAV_2_USI667_Min_Gamma": uav_2_min_nr[2], "UAV_3_USI667_Min_Gamma": uav_3_min_nr[2],
                                 "UAV_4_USI667_Min_Gamma": uav_4_min_nr[2], "UAV_5_USI667_Min_Gamma": uav_5_min_nr[2], "UAV_6_USI667_Min_Gamma": uav_6_min_nr[2], "UAV_7_USI667_Min_Gamma": uav_7_min_nr[2],
                                 "UL_USI667_Min_Gamma": ul_min_nr[2], "VID_USI667_Min_Gamma": vid_min_nr[2],
                                 "UAV_0_USI100_Min_Gamma": uav_0_min_nr[3], "UAV_1_USI100_Min_Gamma": uav_1_min_nr[3], "UAV_2_USI100_Min_Gamma": uav_2_min_nr[3], "UAV_3_USI100_Min_Gamma": uav_3_min_nr[3],
                                 "UAV_4_USI100_Min_Gamma": uav_4_min_nr[3], "UAV_5_USI100_Min_Gamma": uav_5_min_nr[3], "UAV_6_USI100_Min_Gamma": uav_6_min_nr[3], "UAV_7_USI100_Min_Gamma": uav_7_min_nr[3],
                                 "UL_USI100_Min_Gamma": ul_min_nr[3], "VID_USI100_Min_Gamma": vid_min_nr[3]}])

num_reliable_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/results/data_manual_num_reliable_min_max_19122024.csv")

In [5]:
# Get min for throughput
usi_list = [10, 20, 66.7, 100]
ul_min_th = []
uav_0_min_th = []
uav_1_min_th = []
uav_2_min_th = []
uav_3_min_th = []
uav_4_min_th = []
uav_5_min_th = []
uav_6_min_th = []
uav_7_min_th = []
vid_min_th = []
for usi in usi_list:
    ul_usi_df = ul_num_reliable_df_tmp.loc[ul_num_reliable_df_tmp["USI"]==usi]
    ul_min_th.append(ul_usi_df["Throughput"].min())
    dl_df = uav_0_num_reliable_df_tmp.loc[uav_0_num_reliable_df_tmp["USI"]==usi]
    uav_0_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_1_num_reliable_df_tmp.loc[uav_1_num_reliable_df_tmp["USI"]==usi]
    uav_1_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_2_num_reliable_df_tmp.loc[uav_2_num_reliable_df_tmp["USI"]==usi]
    uav_2_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_3_num_reliable_df_tmp.loc[uav_3_num_reliable_df_tmp["USI"]==usi]
    uav_3_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_4_num_reliable_df_tmp.loc[uav_4_num_reliable_df_tmp["USI"]==usi]
    uav_4_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_5_num_reliable_df_tmp.loc[uav_5_num_reliable_df_tmp["USI"]==usi]
    uav_5_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_6_num_reliable_df_tmp.loc[uav_6_num_reliable_df_tmp["USI"]==usi]
    uav_6_min_th.append(dl_df["Throughput"].min())
    dl_df = uav_7_num_reliable_df_tmp.loc[uav_7_num_reliable_df_tmp["USI"]==usi]
    uav_7_min_th.append(dl_df["Throughput"].min())
    vid_df = vid_num_reliable_df_tmp.loc[vid_num_reliable_df_tmp["USI"]==usi]
    vid_min_th.append(vid_df["Throughput"].min())

throughput_df = pd.DataFrame([{"UAV_0_USI10_Min_Throughput": uav_0_min_th[0], "UAV_1_USI10_Min_Throughput": uav_1_min_th[0], "UAV_2_USI10_Min_Throughput": uav_2_min_th[0], "UAV_3_USI10_Min_Throughput": uav_3_min_th[0],
                                 "UAV_4_USI10_Min_Throughput": uav_4_min_th[0], "UAV_5_USI10_Min_Throughput": uav_5_min_th[0], "UAV_6_USI10_Min_Throughput": uav_6_min_th[0], "UAV_7_USI10_Min_Throughput": uav_7_min_th[0],
                                 "UL_USI10_Min_Throughput": ul_min_th[0], "VID_USI10_Min_Throughput": vid_min_th[0],
                                 "UAV_0_USI20_Min_Throughput": uav_0_min_th[1], "UAV_1_USI20_Min_Throughput": uav_1_min_th[1], "UAV_2_USI20_Min_Throughput": uav_2_min_th[1], "UAV_3_USI20_Min_Throughput": uav_3_min_th[1],
                                 "UAV_4_USI20_Min_Throughput": uav_4_min_th[1], "UAV_5_USI20_Min_Throughput": uav_5_min_th[1], "UAV_6_USI20_Min_Throughput": uav_6_min_th[1], "UAV_7_USI20_Min_Throughput": uav_7_min_th[1],
                                 "UL_USI20_Min_Throughput": ul_min_th[1], "VID_USI20_Min_Throughput": vid_min_th[1],
                                 "UAV_0_USI667_Min_Throughput": uav_0_min_th[2], "UAV_1_USI667_Min_Throughput": uav_1_min_th[2], "UAV_2_USI667_Min_Throughput": uav_2_min_th[2], "UAV_3_USI667_Min_Throughput": uav_3_min_th[2],
                                 "UAV_4_USI667_Min_Throughput": uav_4_min_th[2], "UAV_5_USI667_Min_Throughput": uav_5_min_th[2], "UAV_6_USI667_Min_Throughput": uav_6_min_th[2], "UAV_7_USI667_Min_Throughput": uav_7_min_th[2],
                                 "UL_USI667_Min_Throughput": ul_min_th[2], "VID_USI667_Min_Throughput": vid_min_th[2],
                                 "UAV_0_USI100_Min_Throughput": uav_0_min_th[3], "UAV_1_USI100_Min_Throughput": uav_1_min_th[3], "UAV_2_USI100_Min_Throughput": uav_2_min_th[3], "UAV_3_USI100_Min_Throughput": uav_3_min_th[3],
                                 "UAV_4_USI100_Min_Throughput": uav_4_min_th[3], "UAV_5_USI100_Min_Throughput": uav_5_min_th[3], "UAV_6_USI100_Min_Throughput": uav_6_min_th[3], "UAV_7_USI100_Min_Throughput": uav_7_min_th[3],
                                 "UL_USI100_Min_Throughput": ul_min_th[3], "VID_USI100_Min_Throughput": vid_min_th[3]}])

throughput_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/results/data_manual_throughput_min_max_new_19122024.csv")

In [39]:
# Let's try excluding any time window with failure
ul_num_reliable_df_tmp = ul_num_reliable_df.copy()
ul_num_reliable_df_tmp.rename(columns={"Measured_Reliability": "UL_Measured_Reliability", "Measured_Reliability_1": "UL_Measured_Reliability_1", "Num_Reliable": "UL_Num_Reliable"}, inplace=True)
ul_num_reliable_df_tmp["UAV_0_Measured_Reliability"] = uav_0_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_0_Measured_Reliability_1"] = uav_0_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_0_Num_Reliable"] = uav_0_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_1_Measured_Reliability"] = uav_1_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_1_Measured_Reliability_1"] = uav_1_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_1_Num_Reliable"] = uav_1_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_2_Measured_Reliability"] = uav_2_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_2_Measured_Reliability_1"] = uav_2_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_2_Num_Reliable"] = uav_2_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_3_Measured_Reliability"] = uav_3_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_3_Measured_Reliability_1"] = uav_3_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_3_Num_Reliable"] = uav_3_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_4_Measured_Reliability"] = uav_4_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_4_Measured_Reliability_1"] = uav_4_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_4_Num_Reliable"] = uav_4_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_5_Measured_Reliability"] = uav_5_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_5_Measured_Reliability_1"] = uav_5_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_5_Num_Reliable"] = uav_5_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_6_Measured_Reliability"] = uav_6_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_6_Measured_Reliability_1"] = uav_6_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_6_Num_Reliable"] = uav_6_num_reliable_df["Num_Reliable"]
ul_num_reliable_df_tmp["UAV_7_Measured_Reliability"] = uav_7_num_reliable_df["Measured_Reliability"]
ul_num_reliable_df_tmp["UAV_7_Measured_Reliability_1"] = uav_7_num_reliable_df["Measured_Reliability_1"]
ul_num_reliable_df_tmp["UAV_7_Num_Reliable"] = uav_7_num_reliable_df["Num_Reliable"]

ul_num_reliable_df_tmp = ul_num_reliable_df_tmp.loc[(ul_num_reliable_df_tmp["Time"]>=1) & (ul_num_reliable_df_tmp["UL_Measured_Reliability_1"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_0_Measured_Reliability_1"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_1_Measured_Reliability_1"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_2_Measured_Reliability_1"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_3_Measured_Reliability_1"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_4_Measured_Reliability_1"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_5_Measured_Reliability_1"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_6_Measured_Reliability_1"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_7_Measured_Reliability_1"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UL_Measured_Reliability"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_0_Measured_Reliability"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_1_Measured_Reliability"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_2_Measured_Reliability"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_3_Measured_Reliability"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_4_Measured_Reliability"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_5_Measured_Reliability"]>=RELIABILITY_TH) &
                                                    (ul_num_reliable_df_tmp["UAV_6_Measured_Reliability"]>=RELIABILITY_TH) & (ul_num_reliable_df_tmp["UAV_7_Measured_Reliability"]>=RELIABILITY_TH)]

# Get min for DL
dl_min_num_reliable = np.nanmin((ul_num_reliable_df_tmp["UAV_0_Num_Reliable"].min(), ul_num_reliable_df_tmp["UAV_1_Num_Reliable"].min(),
                                ul_num_reliable_df_tmp["UAV_2_Num_Reliable"].min(), ul_num_reliable_df_tmp["UAV_3_Num_Reliable"].min(),
                                ul_num_reliable_df_tmp["UAV_4_Num_Reliable"].min(), ul_num_reliable_df_tmp["UAV_5_Num_Reliable"].min(),
                                ul_num_reliable_df_tmp["UAV_6_Num_Reliable"].min(), ul_num_reliable_df_tmp["UAV_7_Num_Reliable"].min()))
dl_max_num_reliable = np.nanmin((ul_num_reliable_df_tmp["UAV_0_Num_Reliable"].max(), ul_num_reliable_df_tmp["UAV_1_Num_Reliable"].max(),
                                ul_num_reliable_df_tmp["UAV_2_Num_Reliable"].max(), ul_num_reliable_df_tmp["UAV_3_Num_Reliable"].max(),
                                ul_num_reliable_df_tmp["UAV_4_Num_Reliable"].max(), ul_num_reliable_df_tmp["UAV_5_Num_Reliable"].max(),
                                ul_num_reliable_df_tmp["UAV_6_Num_Reliable"].max(), ul_num_reliable_df_tmp["UAV_7_Num_Reliable"].max()))

usi_list = [10, 20, 66.7, 100]
usi_min_num_reliable = []
usi_max_num_reliable = []
for usi in usi_list:
    ul_usi_df = ul_num_reliable_df_tmp.loc[ul_num_reliable_df_tmp["USI"]==usi]
    usi_min_num_reliable.append(ul_usi_df["UL_Num_Reliable"].min())
    usi_max_num_reliable.append(ul_usi_df["UL_Num_Reliable"].max())

num_reliable_df = pd.DataFrame([{"DL_Min_Num_Reliable": dl_min_num_reliable, "UL_USI10_Min_Num_Reliable": usi_min_num_reliable[0], "UL_USI20_Min_Num_Reliable": usi_min_num_reliable[1], 
                                "UL_USI667_Min_Num_Reliable": usi_min_num_reliable[2], "UL_USI100_Min_Num_Reliable": usi_min_num_reliable[3], 
                                "DL_Max_Num_Reliable": dl_max_num_reliable, "UL_USI10_Max_Num_Reliable": usi_max_num_reliable[0], "UL_USI20_Max_Num_Reliable": usi_max_num_reliable[1], 
                                "UL_USI667_Max_Num_Reliable": usi_max_num_reliable[2], "UL_USI100_Max_Num_Reliable": usi_max_num_reliable[3]}])

## Temporary to overcome mem issue

In [11]:
RELIABILITY_TH = 0.999
uav_0_num_reliable_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/DJISpark_Measured_Throughput_100000_Samples/data_manual_throughput_train_vid_processed.csv")
uav_0_num_reliable_df_tmp = uav_0_num_reliable_df.loc[(uav_0_num_reliable_df["Measured_Reliability"] >= RELIABILITY_TH) & (uav_0_num_reliable_df["Measured_Reliability_1"] >= RELIABILITY_TH) & (uav_0_num_reliable_df["Time"] >= 1)]
uav_0_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
usi_list = [10, 20, 66.7, 100]
uav_0_min_nr = []
uav_0_min_th = []

for usi in usi_list:
    dl_df = uav_0_num_reliable_df_tmp.loc[uav_0_num_reliable_df_tmp["USI"]==usi]
    uav_0_min_nr.append(dl_df["Num_Reliable"].min())
    uav_0_min_th.append(dl_df["Throughput"].min())

num_reliable_df = pd.DataFrame([{"UAV_0_USI10_Min_Gamma": uav_0_min_nr[0], "UAV_0_USI20_Min_Gamma": uav_0_min_nr[1], "UAV_0_USI667_Min_Gamma": uav_0_min_nr[2], "UAV_0_USI100_Min_Gamma": uav_0_min_nr[3],
                                 "UAV_0_USI10_Min_Throughput": uav_0_min_th[0], "UAV_0_USI20_Min_Throughput": uav_0_min_th[1], "UAV_0_USI667_Min_Throughput": uav_0_min_th[2], "UAV_0_USI100_Min_Throughput": uav_0_min_th[3]}])

num_reliable_df.to_csv("/home/research-student/omnet-fanet/data-processing-scripts/journal_3_scripts/temp/vid_min_nr_th.csv")

/tmp/ipykernel_1421538/2245203202.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  uav_0_num_reliable_df_tmp.dropna(subset=["Measured_Reliability", "Num_Reliable"], inplace=True)
